In [ ]:
# 과제: 토큰을 가지고 파이썬 코드로 Dynamodb 연동 후 ‘Test’테이블 생성
# 'Test' 테이블 생성까지 진행 완료했습니다 (과제 폴더에 사진 첨부 완료)
# aws key value는 삭제하고 제출드립니다

import boto3
from boto3.dynamodb.conditions import Key, Attr
from decimal import Decimal

def create_robot_table(dynamodb=None):
    if not dynamodb:
        # 로컬 DynamoDB 연결 설정
        dynamodb = boto3.resource('dynamodb',
                            aws_access_key_id="",
                            aws_secret_access_key="",
                            region_name="ap-northeast-2",
                            endpoint_url="https://dynamodb.ap-northeast-2.amazonaws.com")
    
    # 과제(Notion)에 'Test' 테이블 생성하라고 표기됨
    table_name = 'Test'
    
    # 기존에 테이블이 있으면 삭제 
    try:
        table = dynamodb.Table(table_name)
        table.delete()
        # 삭제 완료까지 대기
        table.meta.client.get_waiter('table_not_exists').wait(TableName=table_name)
    except dynamodb.meta.client.exceptions.ResourceNotFoundException:
        pass

    # 'Test' 테이블 생성 (RobotID와 LogDate를 키로 사용)
    table = dynamodb.create_table(
        TableName=table_name,
        KeySchema=[
            {'AttributeName': 'RobotID', 'KeyType': 'HASH'},  # Partition Key
            {'AttributeName': 'LogDate', 'KeyType': 'RANGE'}   # Sort Key
        ],
        AttributeDefinitions=[
            {'AttributeName': 'RobotID', 'AttributeType': 'S'}, # S: 문자열
            {'AttributeName': 'LogDate', 'AttributeType': 'S'}  # S: 문자열
        ],
        ProvisionedThroughput={'ReadCapacityUnits': 10, 'WriteCapacityUnits': 10}
    )
    
    # 생성 완료까지 대기
    table.meta.client.get_waiter('table_exists').wait(TableName=table_name)
    return table

def write_robot_data(robot_id, log_date, battery_level, dynamodb):
    table = dynamodb.Table('Test')
    table.put_item(
        Item={
            'RobotID': robot_id,
            'LogDate': log_date,
            'Battery': Decimal(str(battery_level)) # 전압이나 배터리 잔량
        }
    )
    print(f"Robot Log [{robot_id} | {log_date}] added!")

if __name__ == '__main__':
    # 로컬 DB 리소스 생성
    dynamodb_res = boto3.resource('dynamodb',
                            aws_access_key_id="",
                            aws_secret_access_key="",
                            region_name="ap-northeast-2",
                            endpoint_url="https://dynamodb.ap-northeast-2.amazonaws.com")
    
    # 테이블 생성
    test_table = create_robot_table(dynamodb_res)
    print("Table status:", test_table.table_status)
    
    # 샘플 데이터 입력
    write_robot_data('Kinesis-01', '2026-04-21 15:00', 95.5, dynamodb_res)

Table status: CREATING
Robot Log [Kinesis-01 | 2026-04-21 15:00] added!
